# 🏠 KHS Housing Financial Vulnerability — World-Class EDA
### Kenya Housing Survey · master_hfvs_v4 · Informed Exploratory Data Analysis

**Purpose:** Let the data speak — gradual, step-by-step exploration covering:
1. Dataset topology & data quality audit  
2. Target distribution & binary classification framing  
3. Dimension score analysis (D1–D5)  
4. Demographic & structural correlates  
5. Geographic vulnerability profiling  
6. Multivariate relationships & interaction detection  
7. Feature-level informatics (missingness, skewness, mutual information)  
8. Modelling readiness — what to drop, engineer, and predict  

> Every visual is paired with exact numeric outputs for accuracy of findings.


## ⚙️ Step 0 — Environment Setup & Data Load

In [ ]:

import json, warnings, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from pathlib import Path
from sklearn.feature_selection import mutual_info_classif, mutual_info_regression
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')
np.random.seed(42)

# ── Colour palette ─────────────────────────────────────────────────────────────
TEAL   = '#00695C'; RED    = '#B71C1C'; AMBER  = '#E65100'
BLUE   = '#1565C0'; PURPLE = '#6A1B9A'; GRAY   = '#546E7A'; DARK   = '#2C2C2A'
GREEN  = '#2E7D32'; ORANGE = '#F57C00'

plt.rcParams.update({
    'figure.dpi': 130, 'figure.facecolor': 'white',
    'axes.facecolor': '#F8F8F6', 'axes.spines.top': False,
    'axes.spines.right': False, 'axes.titlesize': 13,
    'axes.titleweight': '600', 'axes.labelsize': 11,
    'xtick.labelsize': 9, 'ytick.labelsize': 9,
    'font.family': 'sans-serif', 'legend.framealpha': 0.9, 'legend.fontsize': 9,
})

# ── Load master dataset ────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

DRIVE = Path('/content/drive/MyDrive/KHS_Dissertation')
MODS  = DRIVE / 'outputs' / 'models'
FIGS  = DRIVE / 'outputs' / 'figures'
FIGS.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(MODS / 'master_hfvs_v4.parquet')

COUNTY_MAP = {
     1:'Mombasa',2:'Kwale',3:'Kilifi',4:'Tana River',5:'Lamu',6:'Taita-Taveta',
     7:'Garissa',8:'Wajir',9:'Mandera',10:'Marsabit',11:'Isiolo',12:'Meru',
    13:'Tharaka-Nithi',14:'Embu',15:'Kitui',16:'Machakos',17:'Makueni',
    18:'Nyandarua',19:'Nyeri',20:'Kirinyaga',21:"Murang'a",22:'Kiambu',
    23:'Turkana',24:'West Pokot',25:'Samburu',26:'Trans Nzoia',27:'Uasin Gishu',
    28:'Elgeyo-Marakwet',29:'Nandi',30:'Baringo',31:'Laikipia',32:'Nakuru',
    33:'Narok',34:'Kajiado',35:'Kericho',36:'Bomet',37:'Kakamega',38:'Vihiga',
    39:'Bungoma',40:'Busia',41:'Siaya',42:'Kisumu',43:'Homa Bay',44:'Migori',
    45:'Kisii',46:'Nyamira',47:'Nairobi',
}
df['county_name'] = df['a01'].map(COUNTY_MAP)

HFVS_THRESHOLD = 0.60
df['hfvs_binary'] = (df['hfvs'] >= HFVS_THRESHOLD).astype(int)

print(f"✓ Master dataset loaded")
print(f"  Shape   : {df.shape[0]:,} rows × {df.shape[1]:,} columns")
print(f"  Memory  : {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"  Counties: {df['a01'].nunique()}")
print(f"  HFVS target col: 'hfvs_binary'  (threshold = {HFVS_THRESHOLD})")


---
## 📐 Step 1 — Dataset Topology & Column Inventory

In [ ]:

# 1A: Column type taxonomy
num_cols  = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols  = df.select_dtypes(include=['object','category']).columns.tolist()
bool_cols = df.select_dtypes(include=['bool']).columns.tolist()

# Separate HFVS formula columns from raw survey columns
formula_cols = ['d1_score','d2_score','d3_score','d4_score','d5_score','hfvs','hfvs_binary',
                'rent_burden','no_savings','no_loan_access',
                'd2_no_land','d2_no_agreement','d2_tenure_sys','d2_doc_insecurity','d2_eviction',
                'd3_flood','d3_mudslide','d3_terrain',
                'd4_wall','d4_roof','d4_floor','d4_overcrowd',
                'd5_water','d5_toilet','d5_electric','d5_cooking',
                'effective_rent','total_exp_proxy']
engineered_cols = [c for c in formula_cols if c in df.columns]
raw_cols = [c for c in df.columns if c not in engineered_cols]

print("=" * 60)
print("DATASET TOPOLOGY")
print("=" * 60)
print(f"  Total columns         : {df.shape[1]:,}")
print(f"  Numeric columns       : {len(num_cols):,}")
print(f"  Categorical columns   : {len(cat_cols):,}")
print(f"  Boolean columns       : {len(bool_cols):,}")
print(f"  ── of which engineered: {len(engineered_cols):,}")
print(f"  ── raw survey cols    : {len(raw_cols):,}")
print(f"\nRow count             : {df.shape[0]:,} households")
print(f"Geographic coverage    : {df['a01'].nunique()} / 47 counties")

# 1B: Survey module breakdown (by column prefix)
from collections import Counter
prefixes = Counter(c[0] if c[0].isalpha() else 'other' for c in raw_cols if len(c) >= 1)
print("\nColumn distribution by survey module prefix:")
for k, v in sorted(prefixes.items(), key=lambda x: -x[1]):
    print(f"  Module '{k}' : {v:>4} columns")


In [ ]:

# 1C: Visualise column taxonomy as a sunburst-style bar
modules = {}
for c in raw_cols:
    if len(c) >= 1 and c[0].isalpha():
        modules.setdefault(c[0], []).append(c)

labels = sorted(modules.keys(), key=lambda k: -len(modules[k]))
counts = [len(modules[k]) for k in labels]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: survey module sizes
colors = plt.cm.tab20(np.linspace(0, 1, len(labels)))
bars = axes[0].barh(labels, counts, color=colors, edgecolor='white', linewidth=0.5)
axes[0].set_title("Raw Survey Columns by Module Prefix")
axes[0].set_xlabel("Number of Columns")
for bar, cnt in zip(bars, counts):
    axes[0].text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
                 str(cnt), va='center', fontsize=8)

# Right: column type pie
type_labels = ['Numeric (raw)', 'Engineered features', 'Categorical', 'Boolean']
type_values = [len(num_cols) - len(engineered_cols), len(engineered_cols),
               len(cat_cols), len(bool_cols)]
wedge_colors = [BLUE, TEAL, AMBER, PURPLE]
axes[1].pie(type_values, labels=type_labels, colors=wedge_colors,
            autopct='%1.1f%%', startangle=140, pctdistance=0.75,
            wedgeprops=dict(edgecolor='white', linewidth=1.5))
axes[1].set_title("Column Type Distribution")

plt.suptitle("Dataset Topology Overview", fontsize=14, fontweight='700', y=1.01)
plt.tight_layout()
plt.savefig(FIGS / 'eda01_topology.png', dpi=130, bbox_inches='tight')
plt.show()

# Numeric summary
print("\n── Column Type Summary ──")
for lbl, val in zip(type_labels, type_values):
    print(f"  {lbl:<25}: {val:>4}")


---
## 🕳️ Step 2 — Missingness Audit

In [ ]:

# Compute missingness across ALL columns
miss = df.isnull().mean() * 100
miss = miss[miss > 0].sort_values(ascending=False)

print("=" * 55)
print("MISSINGNESS AUDIT")
print("=" * 55)
print(f"  Columns with zero nulls  : {(df.isnull().sum() == 0).sum():,}")
print(f"  Columns with ANY nulls   : {(df.isnull().sum() > 0).sum():,}")
print(f"  Columns >50% missing     : {(miss > 50).sum():,}")
print(f"  Columns >80% missing     : {(miss > 80).sum():,}")
print(f"  Rows with zero nulls     : {df.dropna().shape[0]:,}  ({df.dropna().shape[0]/len(df)*100:.1f}%)")

print("\n── Top 30 most-missing columns ──")
print(f"  {'Column':<25} {'% Missing':>10}")
print("  " + "─" * 37)
for col, pct in miss.head(30).items():
    bar = "█" * int(pct / 5)
    print(f"  {col:<25} {pct:>9.1f}%  {bar}")


In [ ]:

# Missingness heatmap — top 40 columns only (readable)
top_miss_cols = miss.head(40).index.tolist()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: sorted bar of top 40
miss40 = miss.head(40)
bar_colors = [RED if v > 50 else AMBER if v > 20 else BLUE for v in miss40.values]
axes[0].barh(range(len(miss40)), miss40.values, color=bar_colors)
axes[0].set_yticks(range(len(miss40)))
axes[0].set_yticklabels(miss40.index, fontsize=7)
axes[0].set_xlabel("% Missing")
axes[0].set_title("Top 40 Columns by Missingness")
axes[0].axvline(50, color=RED, linestyle='--', lw=1, label='>50% missing')
axes[0].axvline(20, color=AMBER, linestyle='--', lw=1, label='>20% missing')
axes[0].legend(fontsize=8)
axes[0].invert_yaxis()

# Right: histogram of missingness rates
miss_all = df.isnull().mean() * 100
axes[1].hist(miss_all[miss_all > 0], bins=30, color=TEAL, edgecolor='white', alpha=0.85)
axes[1].set_xlabel("% Missing per Column")
axes[1].set_ylabel("Number of Columns")
axes[1].set_title("Distribution of Missingness Rates")
thresholds = [20, 50, 80]
for t in thresholds:
    n = (miss_all > t).sum()
    axes[1].axvline(t, color=RED, linestyle=':', lw=1.5)
    axes[1].text(t+1, axes[1].get_ylim()[1]*0.85, f'>{t}%\n({n} cols)',
                 fontsize=7.5, color=RED)

plt.suptitle("Missing Data Landscape", fontsize=14, fontweight='700')
plt.tight_layout()
plt.savefig(FIGS / 'eda02_missingness.png', dpi=130, bbox_inches='tight')
plt.show()

# Print distribution buckets
print("\n── Missingness Bucket Summary ──")
buckets = [(0,1,'< 1%'), (1,5,'1–5%'), (5,20,'5–20%'), (20,50,'20–50%'), (50,80,'50–80%'), (80,100,'> 80%')]
for lo, hi, label in buckets:
    n = ((miss_all > lo) & (miss_all <= hi)).sum()
    print(f"  {label:<10}: {n:>4} columns")


---
## 🎯 Step 3 — Target Variable: HFVS Composite & Binary Label

In [ ]:

print("=" * 55)
print("HFVS CONTINUOUS DISTRIBUTION")
print("=" * 55)
desc = df['hfvs'].describe()
for k, v in desc.items():
    print(f"  {k:<8}: {v:.4f}")

sk = df['hfvs'].skew()
ku = df['hfvs'].kurtosis()
print(f"  Skewness : {sk:.4f}  ({'right-skewed' if sk > 0 else 'left-skewed'})")
print(f"  Kurtosis : {ku:.4f}")

print("\n── Threshold Sensitivity ──")
print(f"  {'Threshold':<12} {'N High-Vuln':>13} {'% HHs':>8}")
print("  " + "─" * 35)
for t in np.arange(0.30, 0.81, 0.05):
    n = (df['hfvs'] >= t).sum()
    pct = n / len(df) * 100
    marker = "  ◄ chosen" if abs(t - 0.60) < 0.001 else ""
    print(f"  {t:.2f}        {n:>13,} {pct:>7.1f}%{marker}")

print("\n── Binary Label (threshold=0.60) ──")
vc = df['hfvs_binary'].value_counts()
print(f"  Class 0 (Low-vulnerability) : {vc[0]:,}  ({vc[0]/len(df)*100:.1f}%)")
print(f"  Class 1 (High-vulnerability): {vc[1]:,}  ({vc[1]/len(df)*100:.1f}%)")
print(f"  Imbalance ratio             : {vc[0]/vc[1]:.2f}:1")


In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# ── Left: HFVS histogram ──────────────────────────────────────────────────────
axes[0].hist(df['hfvs'], bins=60, color=TEAL, edgecolor='white', alpha=0.88)
axes[0].axvline(df['hfvs'].mean(),   color=DARK,   linestyle='--', lw=1.8, label=f"Mean={df['hfvs'].mean():.3f}")
axes[0].axvline(df['hfvs'].median(), color=BLUE,   linestyle=':',  lw=1.8, label=f"Median={df['hfvs'].median():.3f}")
axes[0].axvline(0.60,                color=RED,    linestyle='-',  lw=1.8, label='Threshold=0.60')
axes[0].set_xlabel("HFVS Score")
axes[0].set_ylabel("Count")
axes[0].set_title("HFVS Distribution — All Households")
axes[0].legend()

# Annotate region above threshold
ymax = axes[0].get_ylim()[1]
axes[0].fill_betweenx([0, ymax], 0.60, 1.0, alpha=0.08, color=RED)
axes[0].text(0.78, ymax*0.85, f"High
Vulnerable
{(df['hfvs']>=0.60).mean()*100:.1f}%",
             ha='center', color=RED, fontsize=9, fontweight='bold')

# ── Centre: KDE + normal overlay ─────────────────────────────────────────────
from scipy.stats import norm as spnorm
x_vals = np.linspace(df['hfvs'].min(), df['hfvs'].max(), 300)
mu, sigma = df['hfvs'].mean(), df['hfvs'].std()
axes[1].hist(df['hfvs'], bins=60, density=True, color=TEAL, edgecolor='white', alpha=0.6)
axes[1].plot(x_vals, spnorm.pdf(x_vals, mu, sigma), color=DARK, lw=2, label='Normal fit')
df['hfvs'].plot.kde(ax=axes[1], color=RED, lw=2, label='KDE')
axes[1].axvline(0.60, color=RED, linestyle='--', lw=1.5)
axes[1].set_xlabel("HFVS Score"); axes[1].set_ylabel("Density")
axes[1].set_title("KDE vs Normal Fit")
axes[1].legend()

# ── Right: Binary class bar ────────────────────────────────────────────────────
labels = ['Low (< 0.60)', 'High (≥ 0.60)']
vals   = [df['hfvs_binary'].value_counts()[0], df['hfvs_binary'].value_counts()[1]]
colors = [TEAL, RED]
bars = axes[2].bar(labels, vals, color=colors, edgecolor='white', width=0.5)
for bar, v in zip(bars, vals):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height()+100,
                 f'{v:,}\n({v/sum(vals)*100:.1f}%)', ha='center', fontsize=10, fontweight='bold')
axes[2].set_ylabel("Number of Households")
axes[2].set_title("Binary Class Distribution")
axes[2].set_ylim(0, max(vals)*1.15)

plt.suptitle("HFVS Target Variable — Full Profile", fontsize=14, fontweight='700')
plt.tight_layout()
plt.savefig(FIGS / 'eda03_target.png', dpi=130, bbox_inches='tight')
plt.show()


---
## 📊 Step 4 — HFVS Dimension Score Profiles (D1–D5)

In [ ]:

d_cols = ['d1_score','d2_score','d3_score','d4_score','d5_score']
d_labels = {
    'd1_score': 'D1 Financial Stress',
    'd2_score': 'D2 Tenure Insecurity',
    'd3_score': 'D3 Physical Hazard',
    'd4_score': 'D4 Dwelling Quality',
    'd5_score': 'D5 Utility Deprivation',
}
d_colors = [RED, AMBER, PURPLE, BLUE, TEAL]

print("=" * 70)
print("HFVS DIMENSION SCORE SUMMARY")
print("=" * 70)
print(f"  {'Dimension':<25} {'Mean':>7} {'Median':>8} {'Std':>7} {'Min':>7} {'Max':>7} {'Skew':>7}")
print("  " + "─" * 70)
for col in d_cols:
    s = df[col]
    print(f"  {d_labels[col]:<25} {s.mean():>7.3f} {s.median():>8.3f} {s.std():>7.3f} "
          f"{s.min():>7.3f} {s.max():>7.3f} {s.skew():>7.3f}")
print(f"\n  HFVS composite          {df['hfvs'].mean():>7.3f} {df['hfvs'].median():>8.3f} "
      f"{df['hfvs'].std():>7.3f} {df['hfvs'].min():>7.3f} {df['hfvs'].max():>7.3f} "
      f"{df['hfvs'].skew():>7.3f}")

print("\n── Correlation between dimensions ──")
corr_d = df[d_cols + ['hfvs']].corr()
print(corr_d.round(3).to_string())


In [ ]:

fig = plt.figure(figsize=(16, 10))
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# Individual dimension distributions
for i, (col, color) in enumerate(zip(d_cols, d_colors)):
    r, c = divmod(i, 3)
    ax = fig.add_subplot(gs[r, c])
    ax.hist(df[col], bins=50, color=color, edgecolor='white', alpha=0.85)
    ax.axvline(df[col].mean(), color=DARK, linestyle='--', lw=1.5,
               label=f"Mean={df[col].mean():.3f}")
    ax.set_title(d_labels[col], pad=5)
    ax.set_xlabel("Score (0=best, 1=worst)")
    ax.set_ylabel("Count")
    ax.legend(fontsize=8)

# Bottom right: radar / dimension comparison boxplot
ax6 = fig.add_subplot(gs[1, 2])
box_data = [df[c].dropna().values for c in d_cols]
bp = ax6.boxplot(box_data, patch_artist=True, notch=True,
                  medianprops=dict(color=DARK, lw=2))
for patch, color in zip(bp['boxes'], d_colors):
    patch.set_facecolor(color); patch.set_alpha(0.7)
ax6.set_xticks(range(1, 6))
ax6.set_xticklabels(['D1','D2','D3','D4','D5'], fontsize=10)
ax6.set_ylabel("Score")
ax6.set_title("All Dimensions — Boxplot Comparison")

plt.suptitle("HFVS Dimension Score Profiles", fontsize=15, fontweight='700')
plt.savefig(FIGS / 'eda04_dimensions.png', dpi=130, bbox_inches='tight')
plt.show()

# Numeric: mean by class
print("\n── Dimension Means by HFVS Class ──")
print(f"  {'Dimension':<22} {'Low Vuln':>10} {'High Vuln':>10} {'Δ (gap)':>10}")
print("  " + "─" * 55)
for col in d_cols:
    lo = df[df['hfvs_binary']==0][col].mean()
    hi = df[df['hfvs_binary']==1][col].mean()
    print(f"  {d_labels[col]:<22} {lo:>10.3f} {hi:>10.3f} {hi-lo:>10.3f}")


---
## 🏙️ Step 5 — Urban vs Rural Vulnerability Divide

In [ ]:

# a07_1: 1=Urban, 2=Rural (from notebook metadata)
res_map = {1:'Urban', 2:'Rural'}
df['residence'] = df['a07_1'].map(res_map).fillna('Unknown')

print("=" * 60)
print("URBAN / RURAL BREAKDOWN")
print("=" * 60)
print(f"  {'Area':<12} {'N HHs':>8} {'Mean HFVS':>11} {'≥0.50':>8} {'≥0.60':>8} {'≥0.70':>8}")
print("  " + "─" * 60)
for grp_name, grp in df.groupby('residence'):
    print(f"  {grp_name:<12} {len(grp):>8,} {grp['hfvs'].mean():>11.3f} "
          f"{(grp['hfvs']>=0.50).mean()*100:>7.1f}% "
          f"{(grp['hfvs']>=0.60).mean()*100:>7.1f}% "
          f"{(grp['hfvs']>=0.70).mean()*100:>7.1f}%")

print("\n── Dimension means by residence ──")
print(f"  {'Dimension':<22} {'Urban':>8} {'Rural':>8} {'Δ Rural-Urban':>14}")
print("  " + "─" * 55)
for col in d_cols:
    u = df[df['residence']=='Urban'][col].mean()
    r = df[df['residence']=='Rural'][col].mean()
    print(f"  {d_labels[col]:<22} {u:>8.3f} {r:>8.3f} {r-u:>14.3f}")

# Statistical test
u_vals = df[df['residence']=='Urban']['hfvs'].dropna()
r_vals = df[df['residence']=='Rural']['hfvs'].dropna()
stat, pval = stats.mannwhitneyu(u_vals, r_vals, alternative='two-sided')
print(f"\n  Mann-Whitney U test (Urban vs Rural HFVS):")
print(f"  U = {stat:.0f},  p = {pval:.2e}  ({'*** significant' if pval < 0.001 else 'not significant'})")


In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# ── Left: HFVS KDE by residence ───────────────────────────────────────────────
for res, color in zip(['Urban','Rural'], [BLUE, GREEN]):
    sub = df[df['residence']==res]['hfvs']
    sub.plot.kde(ax=axes[0], label=f"{res} (n={len(sub):,})", color=color, lw=2.5)
axes[0].axvline(0.60, color=RED, linestyle='--', lw=1.5, label='Threshold')
axes[0].set_xlabel("HFVS Score"); axes[0].set_ylabel("Density")
axes[0].set_title("HFVS Distribution by Residence")
axes[0].legend()

# ── Centre: Dimension radar as grouped bar ─────────────────────────────────────
x = np.arange(5); width = 0.35
u_means = [df[df['residence']=='Urban'][c].mean() for c in d_cols]
r_means = [df[df['residence']=='Rural'][c].mean() for c in d_cols]
b1 = axes[1].bar(x - width/2, u_means, width, label='Urban', color=BLUE, alpha=0.85, edgecolor='white')
b2 = axes[1].bar(x + width/2, r_means, width, label='Rural', color=GREEN, alpha=0.85, edgecolor='white')
axes[1].set_xticks(x); axes[1].set_xticklabels(['D1','D2','D3','D4','D5'])
axes[1].set_ylabel("Mean Dimension Score"); axes[1].set_title("D1–D5 Means by Residence")
axes[1].legend()
for bar in list(b1) + list(b2):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                 f'{bar.get_height():.2f}', ha='center', fontsize=7.5)

# ── Right: binary class proportions stacked ───────────────────────────────────
res_cross = pd.crosstab(df['residence'], df['hfvs_binary'], normalize='index') * 100
res_cross.columns = ['Low (<0.60)', 'High (≥0.60)']
res_cross.plot(kind='bar', ax=axes[2], color=[TEAL, RED], edgecolor='white', width=0.5)
axes[2].set_xlabel(""); axes[2].set_ylabel("% of Households")
axes[2].set_title("High vs Low Vulnerability by Residence")
axes[2].legend(title="HFVS Class")
axes[2].set_xticklabels(axes[2].get_xticklabels(), rotation=0)
for p in axes[2].patches:
    axes[2].annotate(f'{p.get_height():.1f}%',
                     (p.get_x() + p.get_width()/2., p.get_height()),
                     ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.suptitle("Urban vs Rural Housing Vulnerability", fontsize=14, fontweight='700')
plt.tight_layout()
plt.savefig(FIGS / 'eda05_urban_rural.png', dpi=130, bbox_inches='tight')
plt.show()


---
## 🗺️ Step 6 — Geographic Vulnerability — County Profiles

In [ ]:

county_agg = (
    df.groupby(['a01', 'county_name'])
    .agg(
        n_hh        = ('hfvs',     'count'),
        hfvs_mean   = ('hfvs',     'mean'),
        hfvs_std    = ('hfvs',     'std'),
        pct_high    = ('hfvs',     lambda x: (x >= 0.60).mean() * 100),
        d1_mean     = ('d1_score', 'mean'),
        d2_mean     = ('d2_score', 'mean'),
        d3_mean     = ('d3_score', 'mean'),
        d4_mean     = ('d4_score', 'mean'),
        d5_mean     = ('d5_score', 'mean'),
    )
    .reset_index()
    .sort_values('hfvs_mean', ascending=False)
)
county_agg['rank'] = range(1, len(county_agg)+1)

print("=" * 95)
print("COUNTY VULNERABILITY RANKING  (1 = most vulnerable)")
print("=" * 95)
print(f"  {'Rk':<4} {'County':<20} {'N':>6} {'HFVS':>7}  {'D1':>6}  {'D2':>6}  {'D3':>6}  {'D4':>6}  {'D5':>6}  {'≥0.60':>7}")
print("  " + "─" * 85)
for _, row in county_agg.iterrows():
    print(f"  {int(row['rank']):<4} {str(row['county_name']):<20} {int(row['n_hh']):>6,} "
          f"{row['hfvs_mean']:>7.3f}  {row['d1_mean']:>6.3f}  {row['d2_mean']:>6.3f}  "
          f"{row['d3_mean']:>6.3f}  {row['d4_mean']:>6.3f}  {row['d5_mean']:>6.3f}  "
          f"{row['pct_high']:>6.1f}%")

print(f"\n  National mean HFVS : {df['hfvs'].mean():.3f}")
print(f"  Most vulnerable    : {county_agg.iloc[0]['county_name']} ({county_agg.iloc[0]['hfvs_mean']:.3f})")
print(f"  Least vulnerable   : {county_agg.iloc[-1]['county_name']} ({county_agg.iloc[-1]['hfvs_mean']:.3f})")
print(f"  Range (max-min)    : {county_agg['hfvs_mean'].max() - county_agg['hfvs_mean'].min():.3f}")


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(18, 10))

# ── Left: ranked horizontal bar ───────────────────────────────────────────────
sorted_agg = county_agg.sort_values('hfvs_mean')
bar_colors = [RED if v >= 0.60 else AMBER if v >= 0.50 else TEAL for v in sorted_agg['hfvs_mean']]
axes[0].barh(range(len(sorted_agg)), sorted_agg['hfvs_mean'],
             color=bar_colors, edgecolor='white', height=0.75)
axes[0].set_yticks(range(len(sorted_agg)))
axes[0].set_yticklabels(sorted_agg['county_name'], fontsize=8)
axes[0].axvline(df['hfvs'].mean(), color=DARK, linestyle='--', lw=1.5,
                label=f"National mean={df['hfvs'].mean():.3f}")
axes[0].axvline(0.60, color=RED, linestyle=':', lw=1.5, label='Threshold=0.60')
axes[0].set_xlabel("Mean HFVS")
axes[0].set_title("County HFVS Ranking — All 47 Counties", pad=10)
axes[0].legend(fontsize=9)

# Colour legend patches
from matplotlib.patches import Patch
legend_elements = [Patch(fc=RED, label='Mean ≥ 0.60'), 
                   Patch(fc=AMBER, label='Mean 0.50–0.60'),
                   Patch(fc=TEAL, label='Mean < 0.50')]
axes[0].legend(handles=legend_elements, fontsize=9, loc='lower right')

# ── Right: dimension driver heatmap (top 20 + bottom 5) ───────────────────────
top20 = county_agg.head(15)
heatmap_data = top20[['d1_mean','d2_mean','d3_mean','d4_mean','d5_mean']].values
sns.heatmap(
    pd.DataFrame(heatmap_data,
                 index=top20['county_name'],
                 columns=['D1 Financial','D2 Tenure','D3 Hazard','D4 Dwelling','D5 Utility']),
    ax=axes[1], annot=True, fmt='.2f', cmap='RdYlGn_r',
    vmin=0, vmax=1, linewidths=0.5, linecolor='white',
    cbar_kws={'label':'Score (0=best, 1=worst)'}
)
axes[1].set_title("Dimension Drivers — Top 15 Most Vulnerable Counties")
axes[1].set_xlabel("")
axes[1].tick_params(axis='y', labelsize=8.5)

plt.suptitle("Geographic Vulnerability Distribution — Kenya 47 Counties",
             fontsize=14, fontweight='700', y=1.01)
plt.tight_layout()
plt.savefig(FIGS / 'eda06_counties.png', dpi=130, bbox_inches='tight')
plt.show()

# Print top 5 + bottom 5
print("\n── Top 5 Most Vulnerable Counties (Key Drivers) ──")
for _, row in county_agg.head(5).iterrows():
    dims = sorted([('D1',row['d1_mean']),('D2',row['d2_mean']),('D3',row['d3_mean']),
                   ('D4',row['d4_mean']),('D5',row['d5_mean'])], key=lambda x: -x[1])
    print(f"  #{int(row['rank'])} {row['county_name']:<20}  HFVS={row['hfvs_mean']:.3f}  "
          f"Main drivers: {dims[0][0]}={dims[0][1]:.2f}, {dims[1][0]}={dims[1][1]:.2f}")

print("\n── Bottom 5 Least Vulnerable Counties ──")
for _, row in county_agg.tail(5).iterrows():
    print(f"  #{int(row['rank'])} {row['county_name']:<20}  HFVS={row['hfvs_mean']:.3f}")


---
## 📈 Step 7 — Key Numeric Feature Distributions

In [ ]:

# Focus on interpretable numeric features most likely to matter
focus_numeric = {
    'total_exp_proxy' : 'Total Expenditure Proxy',
    'effective_rent'  : 'Effective Rent (KES)',
    'rent_burden'     : 'Rent Burden (rent/expenditure)',
    'persons_per_room': 'Persons per Room',
    'n_parcels'       : 'Number of Land Parcels',
    'hh_size'         : 'Household Size',
    'wsvc_water_hours': 'Water Service Hours/day',
    'wsvc_unserved_pct':'% Unserved by Water',
    'mort_demand_avg' : 'County Mortgage Demand (avg)',
}

# Subset to those present
focus_present = {k: v for k, v in focus_numeric.items() if k in df.columns}

print("=" * 65)
print("KEY NUMERIC FEATURE DESCRIPTIVE STATISTICS")
print("=" * 65)
print(f"  {'Feature':<28} {'Mean':>9} {'Median':>9} {'Std':>9} {'Skew':>7} {'% Null':>8}")
print("  " + "─" * 65)
for col, label in focus_present.items():
    s = df[col]
    print(f"  {label:<28} {s.mean():>9.2f} {s.median():>9.2f} "
          f"{s.std():>9.2f} {s.skew():>7.2f} {s.isna().mean()*100:>7.1f}%")


In [ ]:

n_feats = len(focus_present)
ncols = 3
nrows = int(np.ceil(n_feats / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(16, nrows * 3.5))
axes = axes.flatten()

for i, (col, label) in enumerate(focus_present.items()):
    ax = axes[i]
    data = df[col].dropna()
    # Winsorise for display (remove top 1%)
    p99 = data.quantile(0.99)
    data_win = data[data <= p99]
    
    ax.hist(data_win, bins=50, color=TEAL, edgecolor='white', alpha=0.82)
    ax.axvline(data_win.mean(), color=RED, linestyle='--', lw=1.5,
               label=f"Mean={data_win.mean():.2f}")
    ax.axvline(data_win.median(), color=BLUE, linestyle=':', lw=1.5,
               label=f"Med={data_win.median():.2f}")
    ax.set_title(label, fontsize=10)
    ax.set_ylabel("Count")
    ax.legend(fontsize=7.5)
    null_pct = df[col].isna().mean() * 100
    if null_pct > 0:
        ax.text(0.98, 0.92, f"{null_pct:.1f}% null", transform=ax.transAxes,
                ha='right', fontsize=8, color=AMBER)

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Key Numeric Feature Distributions (1st–99th pctile)", fontsize=14, fontweight='700')
plt.tight_layout()
plt.savefig(FIGS / 'eda07_numerics.png', dpi=130, bbox_inches='tight')
plt.show()


---
## 🏷️ Step 8 — Key Categorical Feature Profiles

In [ ]:

# Key categorical/binary columns
cat_focus = {
    'i00'   : 'Land Ownership (0=No, 1=Yes)',
    'a07_1' : 'Urban/Rural (1=Urban, 2=Rural)',
    'c01_1' : 'Main Water Source',
    'c04'   : 'Toilet Facility Type',
    'c10'   : 'Electricity Source',
    'c11'   : 'Cooking Fuel',
    'd03'   : 'Wall Material',
    'd15'   : 'Roof Material',
    'd16'   : 'Floor Material',
}
cat_present = {k: v for k, v in cat_focus.items() if k in df.columns}

print("=" * 65)
print("KEY CATEGORICAL FEATURE PROFILES")
print("=" * 65)
for col, label in cat_present.items():
    vc = df[col].value_counts(dropna=False).head(8)
    print(f"\n  {label} ({col})")
    print(f"  {'Value':<8} {'Count':>8} {'%':>8}")
    print("  " + "─" * 28)
    for val, cnt in vc.items():
        pct = cnt / len(df) * 100
        print(f"  {str(val):<8} {cnt:>8,} {pct:>7.1f}%")


In [ ]:

cols_to_plot = list(cat_present.keys())[:6]
nrows = 2; ncols = 3
fig, axes = plt.subplots(nrows, ncols, figsize=(16, 9))
axes = axes.flatten()

for i, col in enumerate(cols_to_plot):
    ax = axes[i]
    label = cat_present[col]
    vc = df[col].value_counts(dropna=False).head(10)
    bar_c = [RED if v == df[df['hfvs_binary']==1][col].mode().values[0] else TEAL for v in vc.index]
    vc.plot(kind='bar', ax=ax, color=TEAL, edgecolor='white', width=0.7)
    ax.set_title(label, fontsize=9.5)
    ax.set_xlabel("")
    ax.set_ylabel("Count")
    ax.tick_params(axis='x', rotation=45, labelsize=8)
    
    # Overlay HFVS mean per category
    ax2 = ax.twinx()
    means = df.groupby(col)['hfvs'].mean().reindex(vc.index)
    ax2.plot(range(len(means)), means.values, 'o-', color=RED, lw=2, ms=5, label='Mean HFVS')
    ax2.set_ylabel("Mean HFVS", color=RED, fontsize=8)
    ax2.tick_params(axis='y', labelcolor=RED, labelsize=7.5)
    ax2.set_ylim(0, 1)

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Key Categorical Features with Mean HFVS Overlay", fontsize=14, fontweight='700')
plt.tight_layout()
plt.savefig(FIGS / 'eda08_categoricals.png', dpi=130, bbox_inches='tight')
plt.show()


---
## 🔗 Step 9 — Feature Correlation with HFVS

In [ ]:

# Pearson + Spearman correlations of numeric cols vs hfvs
numeric_raw = df.select_dtypes(include=[np.number]).columns.tolist()
# Exclude target and formula ancestors
exclude = set(['hfvs','hfvs_binary','d1_score','d2_score','d3_score','d4_score','d5_score',
               'rent_burden','no_savings','no_loan_access','d2_no_land','d2_no_agreement',
               'd2_tenure_sys','d2_doc_insecurity','d2_eviction','d3_flood','d3_mudslide',
               'd3_terrain','d4_wall','d4_roof','d4_floor','d4_overcrowd',
               'd5_water','d5_toilet','d5_electric','d5_cooking',
               'effective_rent','total_exp_proxy'])

candidates = [c for c in numeric_raw if c not in exclude and df[c].isna().mean() < 0.5]

pearson_corrs = {}
spearman_corrs = {}
for col in candidates:
    valid = df[['hfvs', col]].dropna()
    if len(valid) < 100:
        continue
    pearson_corrs[col]  = valid['hfvs'].corr(valid[col], method='pearson')
    spearman_corrs[col] = valid['hfvs'].corr(valid[col], method='spearman')

p_series = pd.Series(pearson_corrs).sort_values(key=abs, ascending=False)
s_series = pd.Series(spearman_corrs).sort_values(key=abs, ascending=False)

print("=" * 60)
print("TOP 30 FEATURES CORRELATED WITH HFVS")
print("=" * 60)
print(f"  {'Feature':<25} {'Pearson':>10} {'Spearman':>10}")
print("  " + "─" * 48)
for col in p_series.head(30).index:
    print(f"  {col:<25} {p_series.get(col, np.nan):>10.4f} {s_series.get(col, np.nan):>10.4f}")

print(f"\n  Total features assessed: {len(p_series)}")
print(f"  |Pearson| > 0.20       : {(p_series.abs() > 0.20).sum()}")
print(f"  |Pearson| > 0.10       : {(p_series.abs() > 0.10).sum()}")


In [ ]:

top20_cols = p_series.head(20).index.tolist()
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# ── Left: correlation bar ──────────────────────────────────────────────────────
colors = [RED if v > 0 else BLUE for v in p_series.head(20).values]
axes[0].barh(range(20), p_series.head(20).values[::-1],
             color=colors[::-1], edgecolor='white')
axes[0].set_yticks(range(20))
axes[0].set_yticklabels(p_series.head(20).index[::-1], fontsize=8.5)
axes[0].axvline(0, color=DARK, lw=1)
axes[0].set_xlabel("Pearson Correlation with HFVS")
axes[0].set_title("Top 20 Features by |Pearson| Correlation")

# ── Right: Pearson vs Spearman scatter ─────────────────────────────────────────
common = list(set(p_series.index) & set(s_series.index))
px = [p_series[c] for c in common]
sx = [s_series[c] for c in common]
axes[1].scatter(px, sx, alpha=0.5, color=TEAL, s=30, edgecolor='white')
axes[1].plot([-1,1],[-1,1], color=GRAY, linestyle='--', lw=1, label='y=x')
axes[1].set_xlabel("Pearson Correlation")
axes[1].set_ylabel("Spearman Correlation")
axes[1].set_title("Pearson vs Spearman — Linearity Check")
axes[1].axhline(0, color=GRAY, lw=0.5); axes[1].axvline(0, color=GRAY, lw=0.5)

# Annotate extreme points
for col in p_series.head(5).index:
    if col in p_series.index and col in s_series.index:
        axes[1].annotate(col, (p_series[col], s_series[col]), fontsize=7,
                         xytext=(5,5), textcoords='offset points')

axes[1].legend()

plt.suptitle("Feature Correlations with HFVS", fontsize=14, fontweight='700')
plt.tight_layout()
plt.savefig(FIGS / 'eda09_correlations.png', dpi=130, bbox_inches='tight')
plt.show()


---
## 🧠 Step 10 — Mutual Information Feature Ranking

In [ ]:

# MI treats non-linear relationships — critical for tree/boosting models
mi_cols = [c for c in candidates if df[c].isna().mean() < 0.3]
mi_df = df[mi_cols + ['hfvs_binary']].dropna()

# Encode
X_mi = mi_df[mi_cols].copy()
y_mi = mi_df['hfvs_binary']

# Fill remaining NaN with median for MI computation
X_mi = X_mi.fillna(X_mi.median())

mi_scores = mutual_info_classif(X_mi, y_mi, random_state=42)
mi_series = pd.Series(mi_scores, index=mi_cols).sort_values(ascending=False)

print("=" * 55)
print("MUTUAL INFORMATION SCORES vs. hfvs_binary")
print("(Captures non-linear dependencies)")
print("=" * 55)
print(f"  {'Feature':<25} {'MI Score':>10}")
print("  " + "─" * 38)
for col, score in mi_series.head(30).items():
    bar = "█" * int(score * 100)
    print(f"  {col:<25} {score:>10.4f}  {bar}")

print(f"\n  Features with MI > 0.01 : {(mi_series > 0.01).sum()}")
print(f"  Features with MI > 0.05 : {(mi_series > 0.05).sum()}")
print(f"  Features with MI > 0.10 : {(mi_series > 0.10).sum()}")


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# ── Left: top 25 MI bar ───────────────────────────────────────────────────────
top25_mi = mi_series.head(25)
bar_colors = [RED if v > 0.10 else AMBER if v > 0.05 else TEAL for v in top25_mi.values]
axes[0].barh(range(25), top25_mi.values[::-1], color=bar_colors[::-1], edgecolor='white')
axes[0].set_yticks(range(25))
axes[0].set_yticklabels(top25_mi.index[::-1], fontsize=8)
axes[0].set_xlabel("Mutual Information Score")
axes[0].set_title("Top 25 Features by Mutual Information (vs hfvs_binary)")
axes[0].axvline(0.10, color=RED,   linestyle='--', lw=1, label='MI=0.10')
axes[0].axvline(0.05, color=AMBER, linestyle='--', lw=1, label='MI=0.05')
axes[0].legend(fontsize=8.5)

# ── Right: MI vs Pearson (discover non-linear heavyweights) ───────────────────
shared = list(set(mi_series.index) & set(p_series.index))
mi_x = [mi_series[c] for c in shared]
pe_x = [abs(p_series[c]) for c in shared]
axes[1].scatter(pe_x, mi_x, alpha=0.5, color=TEAL, s=30, edgecolor='white')
axes[1].set_xlabel("|Pearson Correlation|")
axes[1].set_ylabel("Mutual Information")
axes[1].set_title("MI vs |Pearson| — Non-linear Feature Spotlight")
axes[1].axvline(0.10, color=RED,   linestyle=':', lw=1)
axes[1].axhline(0.10, color=RED,   linestyle=':', lw=1)

# Annotate top MI features
for col in mi_series.head(8).index:
    if col in shared:
        axes[1].annotate(col, (abs(p_series.get(col,0)), mi_series[col]),
                         fontsize=7, xytext=(4,4), textcoords='offset points')

axes[1].text(0.07, 0.12, "High MI\nLow Pearson\n(Non-linear)",
             fontsize=8, color=PURPLE, ha='center',
             bbox=dict(boxstyle='round,pad=0.3', fc='#EDE7F6', ec=PURPLE, alpha=0.7))

plt.suptitle("Mutual Information Feature Ranking", fontsize=14, fontweight='700')
plt.tight_layout()
plt.savefig(FIGS / 'eda10_mutual_info.png', dpi=130, bbox_inches='tight')
plt.show()


---
## 🔀 Step 11 — Interaction & Multivariate Analysis

In [ ]:

# 11A: Dimension-dimension correlation heatmap
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── Left: full correlation heatmap of dimension sub-components ─────────────────
sub_components = [
    'rent_burden','no_savings','no_loan_access',
    'd2_no_land','d2_tenure_sys','d2_doc_insecurity','d2_eviction',
    'd3_flood','d3_mudslide','d3_terrain',
    'd4_wall','d4_roof','d4_floor','d4_overcrowd',
    'd5_water','d5_toilet','d5_electric','d5_cooking',
    'hfvs'
]
sub_present = [c for c in sub_components if c in df.columns]
corr_sub = df[sub_present].corr()

mask = np.triu(np.ones_like(corr_sub, dtype=bool))
sns.heatmap(corr_sub, ax=axes[0], mask=mask, annot=False, cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, square=True,
            xticklabels=[c[:12] for c in sub_present],
            yticklabels=[c[:12] for c in sub_present],
            linewidths=0.3, cbar_kws={'shrink': 0.8})
axes[0].set_title("Sub-component Correlation Matrix (lower triangle)")
axes[0].tick_params(axis='x', rotation=45, labelsize=7)
axes[0].tick_params(axis='y', labelsize=7)

# ── Right: D-score pairplot summary (correlation scatter matrix simplified) ───
d_corr = df[d_cols + ['hfvs']].corr()
sns.heatmap(d_corr, ax=axes[1], annot=True, fmt='.3f', cmap='RdYlGn_r',
            center=0, vmin=-0.5, vmax=1, linewidths=1, linecolor='white',
            square=True, cbar_kws={'shrink': 0.8})
axes[1].set_title("D1–D5 + HFVS Correlation Matrix")

plt.suptitle("Multivariate Correlation Structure", fontsize=14, fontweight='700')
plt.tight_layout()
plt.savefig(FIGS / 'eda11a_correlation_matrix.png', dpi=130, bbox_inches='tight')
plt.show()

# Print key correlations numerically
print("── D-Score × HFVS Correlations ──")
for col in d_cols:
    r = df[col].corr(df['hfvs'])
    print(f"  {d_labels[col]:<25}: r = {r:.4f}")


In [ ]:

# 11B: Scatter matrix of top 5 MI features + hfvs
top5_mi = mi_series.head(5).index.tolist()
scatter_cols = top5_mi + ['hfvs']
scatter_df = df[scatter_cols].dropna().sample(min(3000, len(df)), random_state=42)

# Map hfvs_binary for colour
scatter_df['_class'] = (scatter_df['hfvs'] >= 0.60).astype(int)

fig = plt.figure(figsize=(14, 12))
n = len(scatter_cols) - 1  # exclude hfvs from grid
for i, ci in enumerate(top5_mi):
    for j, cj in enumerate(top5_mi):
        ax = fig.add_subplot(n, n, i*n + j + 1)
        if i == j:
            scatter_df[scatter_df['_class']==0][ci].plot.kde(ax=ax, color=TEAL, lw=1.5, label='Low')
            scatter_df[scatter_df['_class']==1][ci].plot.kde(ax=ax, color=RED,  lw=1.5, label='High')
            ax.set_xlabel(ci[:12], fontsize=7.5)
            if i == 0: ax.legend(fontsize=7)
        else:
            lo = scatter_df[scatter_df['_class']==0]
            hi = scatter_df[scatter_df['_class']==1]
            ax.scatter(lo[cj], lo[ci], alpha=0.12, s=5, color=TEAL)
            ax.scatter(hi[cj], hi[ci], alpha=0.12, s=5, color=RED)
            ax.set_xlabel(cj[:12], fontsize=7); ax.set_ylabel(ci[:12], fontsize=7)
        ax.tick_params(labelsize=6.5)

plt.suptitle("Scatter Matrix — Top 5 MI Features (Blue=Low Vuln, Red=High Vuln)",
             fontsize=13, fontweight='700', y=1.01)
plt.tight_layout()
plt.savefig(FIGS / 'eda11b_scatter_matrix.png', dpi=130, bbox_inches='tight')
plt.show()


---
## 📉 Step 12 — Skewness, Kurtosis & Outlier Audit

In [ ]:

# Assess distribution shape for all numeric features
skew_kurt = []
for col in candidates:
    s = df[col].dropna()
    if len(s) < 50:
        continue
    skewness = s.skew()
    kurtosis = s.kurtosis()
    iqr = s.quantile(0.75) - s.quantile(0.25)
    n_outliers = ((s < s.quantile(0.25) - 1.5*iqr) | (s > s.quantile(0.75) + 1.5*iqr)).sum()
    skew_kurt.append({
        'col': col, 'skewness': skewness, 'kurtosis': kurtosis,
        'n_outliers': n_outliers, 'pct_outliers': n_outliers/len(s)*100
    })

skew_df = pd.DataFrame(skew_kurt).set_index('col')

print("=" * 65)
print("DISTRIBUTION SHAPE AUDIT — ALL NUMERIC FEATURES")
print("=" * 65)
print(f"  High-skew features (|skew| > 2) : {(skew_df['skewness'].abs() > 2).sum()}")
print(f"  High-skew features (|skew| > 5) : {(skew_df['skewness'].abs() > 5).sum()}")
print(f"  High-kurtosis (> 10)            : {(skew_df['kurtosis'] > 10).sum()}")

print("\n── Top 20 most skewed features ──")
top_skew = skew_df.reindex(skew_df['skewness'].abs().sort_values(ascending=False).index).head(20)
print(f"  {'Feature':<25} {'Skewness':>10} {'Kurtosis':>10} {'% Outliers':>12}")
print("  " + "─" * 60)
for col, row in top_skew.iterrows():
    print(f"  {col:<25} {row['skewness']:>10.2f} {row['kurtosis']:>10.2f} {row['pct_outliers']:>11.1f}%")


In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# ── Left: skewness distribution ───────────────────────────────────────────────
skews = skew_df['skewness']
axes[0].hist(skews.clip(-20, 20), bins=40, color=TEAL, edgecolor='white', alpha=0.85)
axes[0].axvline(2,  color=AMBER, linestyle='--', lw=1.5, label='|skew|=2')
axes[0].axvline(-2, color=AMBER, linestyle='--', lw=1.5)
axes[0].axvline(5,  color=RED,   linestyle='--', lw=1.5, label='|skew|=5')
axes[0].axvline(-5, color=RED,   linestyle='--', lw=1.5)
axes[0].set_xlabel("Skewness (clipped ±20)"); axes[0].set_ylabel("Count")
axes[0].set_title("Distribution of Feature Skewness")
axes[0].legend(fontsize=8)

# ── Centre: skewness vs kurtosis scatter ──────────────────────────────────────
axes[1].scatter(skews.clip(-20,20), skew_df['kurtosis'].clip(0,100),
                alpha=0.4, color=TEAL, s=25)
axes[1].axvline(2, color=RED, linestyle=':', lw=1)
axes[1].axvline(-2, color=RED, linestyle=':', lw=1)
axes[1].axhline(10, color=AMBER, linestyle=':', lw=1)
axes[1].set_xlabel("Skewness"); axes[1].set_ylabel("Kurtosis")
axes[1].set_title("Skewness vs Kurtosis
(clipped for visibility)")

# ── Right: outlier rates histogram ────────────────────────────────────────────
axes[2].hist(skew_df['pct_outliers'], bins=30, color=AMBER, edgecolor='white', alpha=0.85)
axes[2].set_xlabel("% Outliers (IQR method)"); axes[2].set_ylabel("Count")
axes[2].set_title("Distribution of Outlier Rates per Feature")
axes[2].axvline(skew_df['pct_outliers'].mean(), color=RED, linestyle='--',
                lw=1.5, label=f"Mean={skew_df['pct_outliers'].mean():.1f}%")
axes[2].legend()

plt.suptitle("Skewness, Kurtosis & Outlier Audit", fontsize=14, fontweight='700')
plt.tight_layout()
plt.savefig(FIGS / 'eda12_skewness.png', dpi=130, bbox_inches='tight')
plt.show()

print("\n── Features needing log transform (|skew| > 2 & right-skewed) ──")
needs_log = skew_df[(skew_df['skewness'] > 2) & (skew_df['skewness'] < 100)]
for col, row in needs_log.head(15).iterrows():
    print(f"  {col:<25}  skew={row['skewness']:.2f}  → recommend log1p transform")


---
## 🧭 Step 13 — EDA Synthesis: Decisions for Modelling

In [ ]:

print("=" * 70)
print("EDA SYNTHESIS — DATA-DRIVEN DECISIONS FOR MODELLING")
print("=" * 70)

print("""
╔══════════════════════════════════════════════════════════════════╗
║  A. WHAT TO DROP                                                 ║
╚══════════════════════════════════════════════════════════════════╝
  1. FORMULA ANCESTORS: d1_score, d2_score, d3_score, d4_score, d5_score,
     rent_burden, no_savings, no_loan_access, d2_no_land, d2_tenure_sys,
     d2_doc_insecurity, d2_eviction, d3_flood, d3_mudslide, d3_terrain,
     d4_wall, d4_roof, d4_floor, d4_overcrowd, d5_water, d5_toilet,
     d5_electric, d5_cooking, effective_rent, total_exp_proxy
     Reason: directly used to construct hfvs → data leakage

  2. EXTREME MISSINGNESS (>50%): these columns will impute poorly
     and add noise. Drop any column missing >50% of values.

  3. NEAR-ZERO VARIANCE: columns with < 2 unique values carry no info.

  4. COUNTY-LEVEL DUPLICATES: county_code duplicate cols after merge.

╔══════════════════════════════════════════════════════════════════╗
║  B. WHAT TO FEATURE ENGINEER                                     ║
╚══════════════════════════════════════════════════════════════════╝
  1. LOG TRANSFORMS: total_exp_proxy, effective_rent, n_parcels
     (highly right-skewed — log1p stabilises)

  2. INTERACTION TERMS:
     - i00 (land_owner) × hh_size  → stress on non-owners with large families
     - residence (urban/rural) × c01_1 (water source)
     - a01 (county) mean HFVS as geo feature

  3. BINARY FLAGS:
     - 'no_piped_water'  (c01_1 ∉ {1,2})
     - 'open_defecation' (c04 ∈ {8,11,12,13})
     - 'no_electricity'  (c10 ∈ {9,10,12})
     - 'solid_fuel'      (c11 ∈ {9,10,11,12,13})
     - 'overcrowded'     (persons_per_room >= 3)

  4. COUNTY FEATURES:
     - county mean HFVS (leave-one-out to avoid leakage)
     - county-level water hours, mortgage demand

╔══════════════════════════════════════════════════════════════════╗
║  C. MODEL TYPE RECOMMENDATION                                    ║
╚══════════════════════════════════════════════════════════════════╝
  TASK TYPE: Binary Classification (hfvs_binary) + Regression (hfvs)

  PRIMARY RECOMMENDATION: Gradient Boosted Trees
  ─────────────────────────────────────────────
  ✓ XGBoost / LightGBM — Reasons:
    • Many categorical/ordinal codes — trees handle natively
    • Highly skewed features — trees are invariant to monotonic transforms
    • Detected non-linear relationships (high MI, lower Pearson)
    • Moderate imbalance (ratio ~2:1) — easily handled with scale_pos_weight
    • Missing values — LightGBM has native missing value handling
    • 500+ features — built-in feature selection via importance
    • Known to excel on tabular survey data

  SECONDARY RECOMMENDATION: Logistic Regression (regularised)
  ─────────────────────────────────────────────────────────────
  ✓ With log-transformed skewed features + one-hot encoded categoricals
  ✓ Provides interpretable coefficients (policy-facing dissertation)
  ✓ Baseline comparator

  TERTIARY: TabNet / Neural Network
  ──────────────────────────────────
  ✓ Already imported — use for ensemble or sensitivity check

  AVOID: Linear regression on raw features (too many violated assumptions)

  CLASS IMBALANCE STRATEGY:
  ─────────────────────────
  Ratio ~2:1 — mild imbalance. Prefer:
  • scale_pos_weight (XGBoost) or class_weight='balanced'
  • Evaluation metric: ROC-AUC, PR-AUC (not raw accuracy)
  • Use StratifiedKFold (5-fold, already defined as N_FOLDS=5)

  VALIDATION APPROACH:
  ────────────────────
  • Primary CV: StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
  • Group consideration: households within same county → potential leakage
    Consider StratifiedGroupKFold with a01 (county) as group if overfitting
    is observed in regional models.
""")


---
## ✅ Step 14 — EDA Summary Dashboard

In [ ]:

fig = plt.figure(figsize=(18, 12))
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

# 1: HFVS histogram
ax1 = fig.add_subplot(gs[0, 0])
ax1.hist(df['hfvs'], bins=50, color=TEAL, edgecolor='white', alpha=0.85)
ax1.axvline(0.60, color=RED, lw=2, linestyle='--', label='Threshold')
ax1.axvline(df['hfvs'].mean(), color=DARK, lw=1.5, linestyle=':', label=f"Mean={df['hfvs'].mean():.2f}")
ax1.set_title("HFVS Distribution"); ax1.set_xlabel("HFVS"); ax1.legend(fontsize=8)

# 2: D-score means bar
ax2 = fig.add_subplot(gs[0, 1])
d_means = [df[c].mean() for c in d_cols]
ax2.bar(['D1','D2','D3','D4','D5'], d_means, color=d_colors, edgecolor='white')
for i, v in enumerate(d_means):
    ax2.text(i, v+0.005, f'{v:.3f}', ha='center', fontsize=9, fontweight='bold')
ax2.set_ylabel("Mean Score"); ax2.set_title("Dimension Score Means")
ax2.set_ylim(0, max(d_means)*1.15)

# 3: County vulnerability (top 15)
ax3 = fig.add_subplot(gs[0, 2])
top15 = county_agg.head(15)
colors_c = [RED if v >= 0.60 else AMBER for v in top15['hfvs_mean']]
ax3.barh(range(15), top15['hfvs_mean'].values[::-1], color=colors_c[::-1])
ax3.set_yticks(range(15))
ax3.set_yticklabels(top15['county_name'].values[::-1], fontsize=7.5)
ax3.axvline(df['hfvs'].mean(), color=DARK, linestyle='--', lw=1.5)
ax3.set_title("Top 15 Vulnerable Counties")
ax3.set_xlabel("Mean HFVS")

# 4: Urban vs rural HFVS kde
ax4 = fig.add_subplot(gs[1, 0])
for res, color in zip(['Urban','Rural'], [BLUE, GREEN]):
    df[df['residence']==res]['hfvs'].plot.kde(ax=ax4, label=res, color=color, lw=2)
ax4.axvline(0.60, color=RED, linestyle='--', lw=1.5)
ax4.set_title("HFVS by Residence"); ax4.legend(fontsize=9)
ax4.set_xlabel("HFVS")

# 5: Binary class
ax5 = fig.add_subplot(gs[1, 1])
vc = df['hfvs_binary'].value_counts()
bars = ax5.bar(['Low Vuln', 'High Vuln'], [vc[0], vc[1]], color=[TEAL, RED], edgecolor='white')
for bar, v in zip(bars, [vc[0], vc[1]]):
    ax5.text(bar.get_x()+bar.get_width()/2, bar.get_height()+100,
             f'{v:,}\n({v/sum(vc.values)*100:.1f}%)', ha='center', fontsize=10)
ax5.set_title("Binary Class Balance"); ax5.set_ylabel("Count")

# 6: Top 15 MI features
ax6 = fig.add_subplot(gs[1, 2])
top15mi = mi_series.head(15)
ax6.barh(range(15), top15mi.values[::-1], color=PURPLE, edgecolor='white', alpha=0.8)
ax6.set_yticks(range(15))
ax6.set_yticklabels(top15mi.index[::-1], fontsize=8)
ax6.set_xlabel("Mutual Information Score")
ax6.set_title("Top 15 Features by MI")

# 7: Missingness summary
ax7 = fig.add_subplot(gs[2, 0])
miss_all2 = df.isnull().mean() * 100
ax7.hist(miss_all2[miss_all2 > 0], bins=30, color=AMBER, edgecolor='white', alpha=0.85)
ax7.set_xlabel("% Missing"); ax7.set_ylabel("# Columns")
ax7.set_title("Column Missingness Distribution")
ax7.axvline(50, color=RED, lw=1.5, linestyle='--', label='>50%')
ax7.legend()

# 8: Skewness distribution
ax8 = fig.add_subplot(gs[2, 1])
ax8.hist(skew_df['skewness'].clip(-15, 15), bins=35, color=GREEN, edgecolor='white', alpha=0.85)
ax8.axvline(2, color=RED, lw=1.5, linestyle='--')
ax8.axvline(-2, color=RED, lw=1.5, linestyle='--')
ax8.set_xlabel("Skewness"); ax8.set_ylabel("# Features")
ax8.set_title("Feature Skewness Distribution")

# 9: Key stats text box
ax9 = fig.add_subplot(gs[2, 2])
ax9.axis('off')
stats_text = f"""
EDA KEY FACTS

Households     :  {len(df):,}
Counties       :  {df['a01'].nunique()} / 47
Total columns  :  {df.shape[1]:,}
Raw survey cols:  {len(raw_cols):,}
Engineered cols:  {len(engineered_cols):,}

HFVS mean      :  {df['hfvs'].mean():.3f}
HFVS std       :  {df['hfvs'].std():.3f}
High vuln (≥.60): {(df['hfvs']>=0.60).mean()*100:.1f}%

Class balance  :  {df['hfvs_binary'].value_counts()[0]:,} : {df['hfvs_binary'].value_counts()[1]:,}
Imbalance ratio:  {df['hfvs_binary'].value_counts()[0]/df['hfvs_binary'].value_counts()[1]:.2f} : 1

Top MI feature :  {mi_series.index[0]}
Top corr feat  :  {p_series.index[0]}
Cols >50% null :  {(df.isnull().mean() > 0.5).sum():,}
"""
ax9.text(0.05, 0.95, stats_text, transform=ax9.transAxes, fontsize=9,
         verticalalignment='top', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='#F0F4F8', alpha=0.8))
ax9.set_title("Summary Statistics")

plt.suptitle("KHS EDA — Master Summary Dashboard", fontsize=16, fontweight='700', y=1.01)
plt.savefig(FIGS / 'eda14_summary_dashboard.png', dpi=140, bbox_inches='tight')
plt.show()
print("\n✅ Full EDA complete. All figures saved to FIGS directory.")
